In [6]:
%pip install -U torch transformers peft huggingface_hub pyyaml pandas pyarrow accelerate safetensors

Note: you may need to restart the kernel to use updated packages.


In [7]:
import json
import yaml
from pathlib import Path
from typing import Any
import pandas as pd
import torch
import torch.nn.functional as F
from huggingface_hub import snapshot_download
from peft import PeftModel
from safetensors.torch import load_file
from transformers import AutoModelForCausalLM, AutoTokenizer

# Configuration
REPO_ID = "ceselder/qwen3.6-27b-nla-rl"
RL_ITERATION = "iter_000300"
CACHE_DIR = Path("/workspace/huggingface_cache")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

INJECTION_TOKEN_ID = 158983  # Token for Layer 1 hook

# Global references
av_model: Any = None
ar_model: Any = None
ar_value_head: Any = None
av_tokenizer: Any = None
nla_meta: dict[str, Any] = {}
repo_snapshot: Path | None = None

In [8]:
import gc


def initialize_infrastructure() -> None:
    """Download required weights and load metadata without loading model weights."""
    global nla_meta, repo_snapshot
    if repo_snapshot is not None:
        print("Infrastructure paths already initialized in kernel.")
        return

    if DEVICE != "cuda":
        raise RuntimeError("A CUDA GPU is required to load Qwen 27B components.")

    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    print("Downloading weights to persistent cache (Volume Disk)...")
    repo_snapshot = Path(
        snapshot_download(
            repo_id=REPO_ID,
            cache_dir=str(CACHE_DIR),
            allow_patterns=[
                "av_base/*",
                "ar_reconstructor/*",
                f"av_rl_adapters/{RL_ITERATION}/*",
                "nla_meta.yaml",
                "data/*",
            ],
        )
    )
    nla_meta = yaml.safe_load((repo_snapshot / "nla_meta.yaml").read_text())
    print("Infrastructure paths initialized.")


def unload_reconstructor() -> None:
    """Free the AR model and value head from VRAM."""
    global ar_model, ar_value_head
    if ar_model is None and ar_value_head is None:
        print("AR is already unloaded.")
        return

    ar_model = None
    ar_value_head = None
    gc.collect()
    torch.cuda.empty_cache()
    print("AR unloaded. GPU VRAM cleared.")


def load_verbalizer() -> None:
    """Load only the AV model into VRAM."""
    global av_model, av_tokenizer
    if repo_snapshot is None:
        raise RuntimeError("Call initialize_infrastructure() before load_verbalizer().")
    if av_model is not None:
        print("AV already loaded in kernel.")
        return
    if ar_model is not None:
        unload_reconstructor()

    av_base = None
    try:
        av_base_path = repo_snapshot / "av_base"
        adapter_path = repo_snapshot / "av_rl_adapters" / RL_ITERATION
        print("Loading AV Tokenizer & Model (~52 GB)...")
        av_tokenizer = AutoTokenizer.from_pretrained(av_base_path)
        av_base = AutoModelForCausalLM.from_pretrained(
            av_base_path,
            torch_dtype=DTYPE,
            device_map=DEVICE,
            low_cpu_mem_usage=True,
        )
        av_model = PeftModel.from_pretrained(av_base, adapter_path).eval()
        print("AV loaded successfully.")
    except Exception:
        av_model = None
        del av_base
        gc.collect()
        torch.cuda.empty_cache()
        raise


def unload_verbalizer() -> None:
    """Completely free the AV model from VRAM while retaining its tokenizer."""
    global av_model
    if av_model is None:
        print("AV is already unloaded.")
        return

    del av_model
    av_model = None
    gc.collect()
    torch.cuda.empty_cache()
    print("AV unloaded. GPU VRAM cleared.")


def load_reconstructor() -> None:
    """Load the AR model and its separate activation value head into VRAM."""
    global ar_model, ar_value_head, av_tokenizer
    if repo_snapshot is None:
        raise RuntimeError("Call initialize_infrastructure() before load_reconstructor().")
    if ar_model is not None:
        print("AR already loaded in kernel.")
        return
    if av_model is not None:
        unload_verbalizer()

    ar_path = repo_snapshot / "ar_reconstructor"
    print("Loading AR Reconstructor (~33 GB)...")
    if av_tokenizer is None:
        av_tokenizer = AutoTokenizer.from_pretrained(repo_snapshot / "av_base")
    ar_model = AutoModelForCausalLM.from_pretrained(
        ar_path,
        torch_dtype=DTYPE,
        device_map=DEVICE,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    ).eval()

    d_model = int(nla_meta["extraction"]["d_model"])
    ar_value_head = torch.nn.Linear(d_model, d_model, bias=False, device=DEVICE, dtype=DTYPE)
    ar_value_head.load_state_dict(load_file(str(ar_path / "value_head.safetensors"), device=DEVICE))
    ar_value_head.eval()
    print("AR and value head loaded successfully. Ready for evaluation!")


# Download files and load metadata; model weights are loaded in separate stages.
initialize_infrastructure()

Fetching 18 files: 100%|██████████| 18/18 [00:00<00:00, 861.59it/s]

Infrastructure paths initialized.


In [20]:
def _format_template(template_name: str, text: str) -> str:
    templates = nla_meta.get("prompt_templates", {})
    template = templates.get(template_name)
    if not isinstance(template, str):
        available = ", ".join(sorted(templates)) if isinstance(templates, dict) else "none"
        raise KeyError(
            f"nla_meta.yaml does not define prompt_templates.{template_name!r}; "
            f"available templates: {available}."
        )

    injection_char = nla_meta.get("tokens", {}).get("injection_char", "㈜")
    return template.format(
        text=text,
        explanation=text,
        activation=text,
        injection_char=injection_char,
    )


def _tokenize_chat_prompt(prompt: str) -> Any:
    """Apply Qwen's chat template without opening a reasoning block."""
    encoded = av_tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
    )
    input_ids = encoded if isinstance(encoded, torch.Tensor) else encoded["input_ids"]
    if input_ids.ndim == 1:
        input_ids = input_ids.unsqueeze(0)
    return {"input_ids": input_ids.to(DEVICE)}


def _module_by_suffix(model: Any, suffix: str) -> Any:
    for name, module in model.named_modules():
        if name.endswith(suffix):
            return module
    raise RuntimeError(f"Could not find module ending with {suffix!r}.")


def av(activation: torch.Tensor, max_new_tokens: int = 250) -> str:
    """Activation Verbalizer: Injects activation vector at Layer 1 and generates explanation text."""
    if av_model is None or av_tokenizer is None:
        raise RuntimeError("Call load_verbalizer() before running av().")

    prompt = _format_template("actor", "")
    inputs = _tokenize_chat_prompt(prompt)
    input_ids = inputs["input_ids"]

    marker_positions = (input_ids[0] == INJECTION_TOKEN_ID).nonzero(as_tuple=True)[0]
    if len(marker_positions) != 1:
        raise ValueError("AV prompt must contain exactly one injection marker token.")
    marker_position = int(marker_positions.item())

    layer_one = _module_by_suffix(av_model, "model.layers.1")
    activation = activation.to(device=DEVICE, dtype=DTYPE)

    def inject(_module: Any, args: tuple[Any, ...]) -> tuple[Any, ...]:
        hidden_states = args[0].clone()
        if marker_position >= hidden_states.shape[1]:
            return args
        scale = hidden_states[:, marker_position, :].norm(dim=-1, keepdim=True)
        replacement = activation / activation.norm(dim=-1, keepdim=True).clamp_min(torch.finfo(activation.dtype).eps)
        hidden_states[:, marker_position, :] = replacement * scale
        return (hidden_states, *args[1:])

    handle = layer_one.register_forward_pre_hook(inject)
    try:
        with torch.inference_mode():
            generated = av_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
            )
    finally:
        handle.remove()

    generated_tokens = generated[0, input_ids.shape[1]:]
    return av_tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


def ar(description: str) -> torch.Tensor:
    """Activation Reconstructor: Maps explanation text back into a reconstructed vector."""
    if ar_model is None or av_tokenizer is None or ar_value_head is None:
        raise RuntimeError("Call load_reconstructor() before running ar().")

    prompt = _format_template("critic", description)
    inputs = _tokenize_chat_prompt(prompt)

    with torch.inference_mode():
        output = ar_model(
            **inputs,
            output_hidden_states=True,
            return_dict=True,
        )
        hidden_states = output.hidden_states
        if not hidden_states:
            raise RuntimeError("AR output did not include hidden states.")
        reconstructed = ar_value_head(hidden_states[-1][:, -1, :])

    return reconstructed.squeeze()


def calculate_mse(altered_text_description: str, original_activation: torch.Tensor) -> float:
    """Reconstruct an activation from text and calculate MSE against the original vector."""
    reconstructed = ar(altered_text_description)
    original = original_activation.to(device=reconstructed.device, dtype=reconstructed.dtype)
    return float(F.mse_loss(reconstructed, original).item())

In [21]:
# Load sample activations
if repo_snapshot is None:
    initialize_infrastructure()
data_path = repo_snapshot / "data" / "example_activations.parquet"
df = pd.read_parquet(data_path)

# Process only activation number one (the first row, index 0).
sample_idx = 0
A = torch.tensor(df.iloc[sample_idx]["activation_vector"], dtype=DTYPE, device=DEVICE)

# Phase 1: generate and persist one verbalization while AV is in VRAM.
load_verbalizer()
try:
    sample_text = av(A)
    verbalizations = [sample_text]
    verbalizations_path = CACHE_DIR / "example_verbalizations.json"
    verbalizations_path.write_text(json.dumps(verbalizations, indent=2))
    print(f"Saved activation {sample_idx + 1} verbalization to {verbalizations_path}")
finally:
    unload_verbalizer()

print(f"--- Original Verbalization ---\n{sample_text}\n")


AR unloaded. GPU VRAM cleared.
Loading AV Tokenizer & Model (~52 GB)...


Loading weights: 100%|██████████| 851/851 [00:06<00:00, 130.29it/s]


AV loaded successfully.
Saved activation 1 verbalization to /workspace/huggingface_cache/example_verbalizations.json
AV unloaded. GPU VRAM cleared.
--- Original Verbalization ---
<explanation>
Product listing/catalog format with fragmented SEO-style entries mixing equipment specs, company names, and unrelated items — typical of scraped industrial equipment web content.

The text has shifted from coherent product descriptions into a list of loosely related Indian stone/cement industry terms and links, suggesting continuation of similar fragmented entries or search-result-style items.

"price list for" is an incomplete phrase beginning a new list item or search query, immediately requiring a noun phrase specifying what product or service (e.g., "stone crusher," "cement," "ready mix") follows.
"
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br

In [26]:
# Phase 2: load AR and score the saved text without loading AV again.
if repo_snapshot is None:
    initialize_infrastructure()

sample_idx = 0
if "df" not in globals():
    data_path = repo_snapshot / "data" / "example_activations.parquet"
    df = pd.read_parquet(data_path)
if "A" not in globals():
    A = torch.tensor(df.iloc[sample_idx]["activation_vector"], dtype=DTYPE, device=DEVICE)

sample_input_text = df.iloc[sample_idx]["detokenized_text_truncated"]
verbalizations_path = CACHE_DIR / "example_verbalizations.json"
if "sample_text" not in globals():
    if not verbalizations_path.exists():
        raise FileNotFoundError(
            "No saved verbalization found. Run the Phase 1 cell successfully first."
        )
    saved_verbalizations = json.loads(verbalizations_path.read_text())
    sample_text = saved_verbalizations[sample_idx]

load_reconstructor()
original_mse = calculate_mse(sample_text, A)
print(
    # f"Input sample:\n{sample_input_text}\n\n"
    f"Original MSE: {original_mse:.6f}"
)

# Reuse the loaded AR model for pruning experiments.
pruned_text = sample_text.replace("SEO-style", "")
pruned_mse = calculate_mse(pruned_text, A)
print(f"Pruned MSE:   {pruned_mse:.6f}")

AR already loaded in kernel.
Original MSE: 0.855469
Pruned MSE:   0.832031
